# Top-down con el share **modelado**

## Por qué este notebook existe

El backtest de `03_Techo_categoria_y_shares` comparó:

```
bottom-up : MM3 del tn
top-down  : MM3 del total  ×  MM3 del share
```

y dio empate. Pero esa comparación **subestima al top-down**, porque usa el modelo
de share más pobre posible: una media móvil asume que la redistribución de share es
ruido alrededor de un promedio.

No lo es. La redistribución depende de cosas observables:

- **si el producto es nuevo** — un lanzamiento gana share mes a mes durante su rampa
- **en qué fase está** — crecimiento, meseta o declive
- **cuántos competidores entraron** a su categoría hace poco
- **la inercia del propio share** y su tendencia

Y hay evidencia previa de que esa estructura existe: `01_EDA_series` mostró que los
productos nacidos dentro de la ventana siguen una curva de vida típica, y
`02_DTW_clusters` que esas formas se repiten desfasadas entre productos.

## El test bien hecho

| | cómo predice `tn_p(t+2)` |
|---|---|
| **bottom-up** | un LightGBM que predice `tn` directo, con todas las features |
| **top-down** | modelo del total de `cat3` × **modelo del share**, con las mismas features |

Mismo algoritmo, mismas features, misma partición temporal, **misma semilla y modo
determinístico**. La única diferencia es la descomposición.

Ese último punto no es un detalle. Hicieron falta **dos** arreglos para que dos
corridas idénticas dieran el mismo número:

1. `deterministic=True` en LightGBM.
2. **Redondear las sumas agregadas.** `group_by().sum()` sobre floats no es
   bit-reproducible: la suma en punto flotante no es asociativa y el orden en que los
   hilos acumulan cambia entre corridas. Las diferencias son de 1e-15, pero se
   amplifican cuando un split del árbol cae sobre un empate cercano, y terminan
   moviendo el WAPE en el tercer decimal — el mismo orden de magnitud que las
   diferencias entre modelos que queremos medir.

Vale la pena tenerlo presente para el pipe: `01_Preprocesamiento` también agrega con
`sum()`, así que dos corridas del pipe entero pueden no dar exactamente lo mismo.

## La ventaja estructural del top-down

Los shares de una categoría **suman 1**. Es una restricción que el bottom-up no
tiene: si predice de más en todos los productos de una categoría, nada lo corrige.
El top-down puede **renormalizar** los shares predichos para que sumen 1, y así el
total queda anclado al modelo del agregado, que es el más confiable (`H1`: 52% menos
error al agregar).

Se reportan las dos versiones, con y sin renormalizar, para aislar cuánto aporta la
restricción por sí sola.

## 0 — Ambiente

In [ ]:
import os, json, warnings
from pathlib import Path

import numpy as np
import polars as pl
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


def resolver_bucket() -> Path:
    env = os.environ.get("LABO3_BUCKET")
    if env:
        return Path(env).expanduser().resolve()
    # ~/buckets/b1 primero: es donde lo monta la instalacion de la catedra,
    # y sirve para cualquier usuario (ds, natalialabo3, el que sea).
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1",
                 "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError("No encontre el bucket. Defini LABO3_BUCKET.")


BUCKET  = resolver_bucket()
DIR_RAW = BUCKET / "datasets"
DIR_OUT = BUCKET / "datasets_fe"
DIR_OUT.mkdir(parents=True, exist_ok=True)

SERIE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
         "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
TINTA, TINTA2, MUDO = "#0b0b0b", "#52514e", "#898781"
GRILLA, EJE_C, FONDO = "#e1e0d9", "#c3c2b7", "#fcfcfb"

plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO,
    "axes.edgecolor": EJE_C, "axes.labelcolor": TINTA2,
    "text.color": TINTA, "xtick.color": MUDO, "ytick.color": MUDO,
    "grid.color": GRILLA, "grid.linewidth": .8,
    "axes.grid": True, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 9, "axes.titlesize": 10, "figure.dpi": 110,
    "legend.frameon": False,
})

def limpiar(ax, titulo=None, y=None, x=None):
    if titulo: ax.set_title(titulo, color=TINTA, loc="left", pad=10)
    if y: ax.set_ylabel(y)
    if x: ax.set_xlabel(x)
    ax.grid(axis="x", visible=False)
    return ax

H = 2                                   # horizonte, igual que el pipe
MESES_TRAIN_FIN = 201905                # mismos cortes que 03_Optuna
MESES_VAL       = [201907, 201908]
MESES_TEST      = [201910]
SEMILLA = 102191
print(f"BUCKET: {BUCKET}")

## 1 — Panel con features de ciclo de vida

Todas las features son **causales**: se calculan con información disponible hasta el
mes de la fila. Nada de picos globales, largo de vida total ni fechas de muerte —
esos fueron los que marqué como leakage silencioso en `01_EDA_series`.

La fase se recalcula acá de forma **expansiva**: el pico es el máximo *hasta ese
mes*, no el de toda la serie.

In [ ]:
sell = pl.read_csv(DIR_RAW / "sell-in.txt.gz", separator="\t")
prod = pl.read_csv(DIR_RAW / "tb_productos.txt", separator="\t").unique(subset=["product_id"])

def a_m(c): return (pl.col(c) // 100) * 12 + (pl.col(c) % 100)
def m_a_periodo(m): return ((m - 1) // 12) * 100 + ((m - 1) % 12) + 1
def periodo_a_m(p): return (p // 100) * 12 + (p % 100)

# .round(6): group_by().sum() en float NO es bit-reproducible -- el orden en que
# los hilos suman cambia entre corridas y con el los ultimos bits. Esas diferencias
# de 1e-15 se amplifican cuando un split del arbol cae en un empate, y terminan
# moviendo el WAPE en el tercer decimal. 6 decimales en toneladas es ruido puro.
base = (sell.group_by(["product_id","periodo"]).agg(pl.col("tn").sum().round(6).alias("tn"))
            .with_columns(a_m("periodo").alias("m")))
nac  = base.group_by("product_id").agg(pl.col("m").min().alias("m_nace"),
                                       pl.col("m").max().alias("m_ult"))
grilla = (nac.select("product_id","m_nace","m_ult")
             .with_columns(pl.int_ranges("m_nace", pl.col("m_ult")+1).alias("m"))
             .explode("m").drop("m_nace","m_ult"))

P = (grilla.join(base, on=["product_id","m"], how="left")
           .with_columns(pl.col("tn").fill_null(0.0))
           .join(prod.select("product_id","cat1","cat2","cat3","brand"), on="product_id", how="left")
           .join(nac.select("product_id","m_nace"), on="product_id", how="left")
           .with_columns((pl.col("m") - pl.col("m_nace")).alias("edad"))
           .sort(["product_id","m"]))

# ── total de la categoria y share ────────────────────────────────────────
CT = P.group_by(["cat3","m"]).agg(pl.col("tn").sum().round(6).alias("tn_cat3"),
                                  pl.col("product_id").n_unique().alias("n_comp"))
P = (P.join(CT, on=["cat3","m"], how="left")
      .with_columns(pl.when(pl.col("tn_cat3") > 0)
                      .then(pl.col("tn")/pl.col("tn_cat3"))
                      .otherwise(0.0).alias("share")))

print(f"panel: {P.height:,} filas · {P['product_id'].n_unique()} productos · {P['cat3'].n_unique()} cat3")

In [ ]:
# ── Features causales a nivel producto ───────────────────────────────────
P = P.sort(["product_id","m"]).with_columns([
    *[pl.col("tn").shift(k).over("product_id").alias(f"tn_lag{k}") for k in range(1, 7)],
    *[pl.col("share").shift(k).over("product_id").alias(f"share_lag{k}") for k in range(1, 7)],
    pl.col("tn").rolling_mean(3, min_periods=1).over("product_id").alias("tn_ma3"),
    pl.col("tn").rolling_mean(12, min_periods=3).over("product_id").alias("tn_ma12"),
    pl.col("tn").rolling_std(12, min_periods=3).over("product_id").alias("tn_sd12"),
    pl.col("share").rolling_mean(3, min_periods=1).over("product_id").alias("share_ma3"),
    pl.col("share").rolling_mean(6, min_periods=2).over("product_id").alias("share_ma6"),
    pl.col("tn").cum_max().over("product_id").alias("pico_hist"),
    ((pl.col("m") - 1) % 12 + 1).alias("mes_cal"),
])

P = P.with_columns([
    # z contra la media movil larga: la senial de reversion mas fuerte del notebook 04
    ((pl.col("tn") - pl.col("tn_ma12")) /
      pl.when(pl.col("tn_sd12") > 0).then(pl.col("tn_sd12")).otherwise(1.0)).alias("z_vs_ma12"),
    (pl.col("share") - pl.col("share_ma6")).alias("share_desvio"),
    (pl.col("share") / pl.when(pl.col("share_ma6") > 0).then(pl.col("share_ma6"))
                         .otherwise(1.0) - 1.0).alias("share_mom"),
    (pl.col("tn") / pl.when(pl.col("pico_hist") > 0).then(pl.col("pico_hist"))
                      .otherwise(1.0)).alias("ratio_pico_hist"),
    (pl.col("edad") <= 6).cast(pl.Int8).alias("es_nuevo"),
    (pl.col("edad") <= 12).cast(pl.Int8).alias("es_reciente"),
])

# ── Fase expansiva (sin mirar el futuro) ─────────────────────────────────
P = P.with_columns(
    ((pl.col("tn_ma3") - pl.col("tn_ma3").shift(3).over("product_id")) /
      pl.when(pl.col("pico_hist") > 0).then(pl.col("pico_hist")).otherwise(1.0)).alias("pend3")
)
U = 0.10
P = P.with_columns(
    pl.when(pl.col("edad") <= 2).then(pl.lit(0))                       # lanzamiento
     .when(pl.col("pend3") > U).then(pl.lit(1))                        # crecimiento
     .when(pl.col("pend3") < -U).then(pl.lit(3))                       # caida
     .otherwise(pl.lit(2))                                             # meseta
     .alias("fase")
)

# ── Features de contexto competitivo ─────────────────────────────────────
# Cuantos productos nacieron en la cat3 en los ultimos 6 meses (presion de entrada)
nuevos = (P.filter(pl.col("edad") == 0).group_by(["cat3","m"])
            .agg(pl.len().alias("entradas")))
ent = (CT.select("cat3","m").join(nuevos, on=["cat3","m"], how="left")
         .with_columns(pl.col("entradas").fill_null(0))
         .sort(["cat3","m"])
         .with_columns(pl.col("entradas").rolling_sum(6, min_periods=1).over("cat3")
                         .alias("entradas_6m")))
P = P.join(ent.select("cat3","m","entradas_6m"), on=["cat3","m"], how="left").sort(["product_id","m"])

# lags del total de la categoria
CT2 = CT.sort(["cat3","m"]).with_columns([
    *[pl.col("tn_cat3").shift(k).over("cat3").alias(f"cat3_lag{k}") for k in (1,2,3,6,12)],
    pl.col("tn_cat3").rolling_mean(3, min_periods=1).over("cat3").alias("cat3_ma3"),
    pl.col("tn_cat3").rolling_mean(12, min_periods=3).over("cat3").alias("cat3_ma12"),
])
P = P.join(CT2.drop("n_comp"), on=["cat3","m"], how="left").sort(["product_id","m"])

# ── Targets ──────────────────────────────────────────────────────────────
# IMPRESCINDIBLE ordenar antes de cualquier shift(): los joins de arriba no
# preservan el orden de las filas, y shift().over() trabaja sobre el orden actual,
# no sobre el mes. Sin este sort el target sale de una fila arbitraria del producto.
P = P.sort(["product_id", "m"])

# Y el shift asume que la serie es densa mensual dentro de la vida del producto
# (garantizado por la densificacion de arriba): shift de H filas == H meses.
_chk = (P.group_by("product_id")
          .agg((pl.col("m").diff().drop_nulls() != 1).sum().alias("saltos"))
          .filter(pl.col("saltos") > 0))
assert _chk.height == 0, f"{_chk.height} productos con huecos temporales: el shift seria incorrecto"

P = P.with_columns([
    pl.col("tn").shift(-H).over("product_id").alias("y_tn"),
    pl.col("share").shift(-H).over("product_id").alias("y_share"),
])
P = P.with_columns(pl.col("m").map_elements(m_a_periodo, return_dtype=pl.Int64).alias("periodo"))
print(f"panel con features: {P.height:,} filas x {P.width} columnas")

## 2 — Partición temporal

In [ ]:
m_tr = periodo_a_m(MESES_TRAIN_FIN)
m_val = [periodo_a_m(p) for p in MESES_VAL]
m_test = [periodo_a_m(p) for p in MESES_TEST]

# Gap obligatorio: una fila de mes t tiene target en t+H
assert min(m_val) - m_tr >= H, "falta gap entre train y val"
assert min(m_test) - max(m_val) >= H, "falta gap entre val y test"

D = P.drop_nulls(["y_tn", "y_share"])
tr = D.filter(pl.col("m") <= m_tr)
va = D.filter(pl.col("m").is_in(m_val))
te = D.filter(pl.col("m").is_in(m_test))

print(f"train {tr.height:>7,} filas   hasta {MESES_TRAIN_FIN}")
print(f"val   {va.height:>7,} filas   {MESES_VAL}")
print(f"test  {te.height:>7,} filas   {MESES_TEST}")
print(f"gap train->val: {min(m_val)-m_tr} meses   val->test: {min(m_test)-max(m_val)} meses")

FEATS = [c for c in D.columns if c not in (
    "product_id","cat3","cat1","cat2","brand","m","periodo","m_nace",
    "y_tn","y_share","tn","share","tn_cat3")]
print(f"\n{len(FEATS)} features: {FEATS}")

def a_pd(df):
    x = df.select(FEATS).to_pandas()
    return x

# deterministic + force_row_wise: sin esto LightGBM multihilo NO es reproducible.
# Dos corridas identicas dan WAPE distintos, y las diferencias entre modelos quedan
# tapadas por el ruido del scheduler. Cuesta algo de velocidad y lo vale.
PARAMS = dict(objective="regression", metric="mae", verbosity=-1, n_jobs=-1,
              seed=SEMILLA, deterministic=True, force_row_wise=True,
              n_estimators=400, learning_rate=.05,
              num_leaves=63, min_child_samples=30,
              subsample=.8, subsample_freq=1, colsample_bytree=.8)

## 3 — Los dos modelos

Uno predice `tn` directo. El otro predice `share`, y por separado se predice el
total de la categoría. Todo lo demás es idéntico.

Para el total de `cat3` se prueban dos opciones y se elige **en validación**: la
media móvil de 3 meses (que ya sabemos que es difícil de batir en un agregado
estable) y un LightGBM sobre el panel de categorías.

In [ ]:
def wape(real, pred):
    real = np.asarray(real, float); pred = np.maximum(np.asarray(pred, float), 0)
    den = np.abs(real).sum()
    return float(np.abs(real - pred).sum() / den) if den else np.nan

# ── Modelo 1: bottom-up, predice tn ──────────────────────────────────────
m_bu = lgb.LGBMRegressor(**PARAMS)
m_bu.fit(a_pd(tr), tr["y_tn"].to_numpy())
pred_bu_va = m_bu.predict(a_pd(va))
pred_bu_te = m_bu.predict(a_pd(te))
print(f"bottom-up   val {wape(va['y_tn'], pred_bu_va):.4f}   test {wape(te['y_tn'], pred_bu_te):.4f}")

# ── Modelo 2a: share ─────────────────────────────────────────────────────
m_sh = lgb.LGBMRegressor(**PARAMS)
m_sh.fit(a_pd(tr), tr["y_share"].to_numpy())
sh_va = np.clip(m_sh.predict(a_pd(va)), 0, None)
sh_te = np.clip(m_sh.predict(a_pd(te)), 0, None)
print(f"modelo de share  val WAPE(share) {wape(va['y_share'], sh_va):.4f}")

# baseline de share para comparar: MM3 (lo que usaba el notebook 03)
sh_va_mm3 = va["share_ma3"].to_numpy(); sh_te_mm3 = te["share_ma3"].to_numpy()
print(f"share MM3        val WAPE(share) {wape(va['y_share'], sh_va_mm3):.4f}")

In [ ]:
# ── Modelo 2b: total de la categoria ─────────────────────────────────────
C = (CT2.sort(["cat3", "m"])          # mismo motivo que arriba: shift necesita orden
        .with_columns(pl.col("tn_cat3").shift(-H).over("cat3").alias("y_cat3"),
                      ((pl.col("m")-1) % 12 + 1).alias("mes_cal"))
        .drop_nulls("y_cat3"))
FC = [c for c in C.columns if c not in ("cat3","m","y_cat3","tn_cat3")]
c_tr = C.filter(pl.col("m") <= m_tr)
c_va = C.filter(pl.col("m").is_in(m_val))
c_te = C.filter(pl.col("m").is_in(m_test))

m_ct = lgb.LGBMRegressor(**{**PARAMS, "n_estimators": 200, "num_leaves": 15})
m_ct.fit(c_tr.select(FC).to_pandas(), c_tr["y_cat3"].to_numpy())

tot_gbm_va = np.clip(m_ct.predict(c_va.select(FC).to_pandas()), 0, None)
tot_mm3_va = c_va["cat3_ma3"].to_numpy()
w_gbm = wape(c_va["y_cat3"], tot_gbm_va); w_mm3 = wape(c_va["y_cat3"], tot_mm3_va)
print(f"total cat3   val WAPE   GBM {w_gbm:.4f}   MM3 {w_mm3:.4f}")

USAR_GBM = w_gbm < w_mm3
print(f"-> se usa {'GBM' if USAR_GBM else 'MM3'} para el total")

def total_pred(cdf):
    return (np.clip(m_ct.predict(cdf.select(FC).to_pandas()), 0, None) if USAR_GBM
            else cdf["cat3_ma3"].to_numpy())

tot_va = dict(zip(zip(c_va["cat3"], c_va["m"]), total_pred(c_va)))
tot_te = dict(zip(zip(c_te["cat3"], c_te["m"]), total_pred(c_te)))

## 4 — Reconstrucción y renormalización

`pred_tn = total_predicho × share_predicho`.

La renormalización divide cada share predicho por la suma de los shares predichos de
su categoría en ese mes, forzando que sumen 1. Es la restricción de *adding-up*: el
total queda anclado al modelo del agregado y los shares sólo deciden el reparto.

In [ ]:
def reconstruir(df, sh, totales, renormalizar):
    key = list(zip(df["cat3"].to_list(), df["m"].to_list()))
    tot = np.array([totales.get(k, np.nan) for k in key])
    s = np.asarray(sh, float).copy()
    if renormalizar:
        suma = {}
        for k, v in zip(key, s):
            suma[k] = suma.get(k, 0.0) + v
        s = np.array([v / suma[k] if suma[k] > 0 else 0.0 for k, v in zip(key, s)])
    out = tot * s
    return np.where(np.isfinite(out), out, 0.0)

filas = []
for nombre, sv, st, renorm in [
    ("top-down (share MM3)",        sh_va_mm3, sh_te_mm3, False),
    ("top-down (share modelado)",   sh_va,     sh_te,     False),
    ("top-down (modelado + renorm)", sh_va,    sh_te,     True),
]:
    pv = reconstruir(va, sv, tot_va, renorm)
    pt = reconstruir(te, st, tot_te, renorm)
    filas.append({"modelo": nombre,
                  "wape_val": round(wape(va["y_tn"], pv), 4),
                  "wape_test": round(wape(te["y_tn"], pt), 4)})

filas.insert(0, {"modelo": "bottom-up (GBM sobre tn)",
                 "wape_val": round(wape(va["y_tn"], pred_bu_va), 4),
                 "wape_test": round(wape(te["y_tn"], pred_bu_te), 4)})
filas.append({"modelo": "naive (repetir t)",
              "wape_val": round(wape(va["y_tn"], va["tn"]), 4),
              "wape_test": round(wape(te["y_tn"], te["tn"]), 4)})

R = pl.DataFrame(filas).with_columns(
    (pl.col("wape_test") - pl.col("wape_val")).round(4).alias("brecha"))
print(R)

# Diagnostico de la renormalizacion: solo tiene sentido si el conjunto de productos
# que predecimos cubre toda la categoria. Si faltan productos (poca historia, target
# nulo), forzar que los shares sumen 1 sobre un subconjunto los infla.
cob = (te.group_by(["cat3","m"]).agg(pl.col("share").sum().alias("share_cubierto")))
print(f"cobertura del share por categoria-mes en test: "
      f"mediana {cob['share_cubierto'].median():.3f}, "
      f"minimo {cob['share_cubierto'].min():.3f}")
print("Si es bastante menor que 1, la renormalizacion infla las predicciones:")
print("reparte el total entero entre los productos que si estan, ignorando a los que")
print("faltan. Por eso puede EMPEORAR aunque la restriccion sea correcta en teoria.")
print()

def w_de(nombre):
    return R.filter(pl.col("modelo") == nombre)["wape_test"][0]

bu      = w_de("bottom-up (GBM sobre tn)")
td_mm3  = w_de("top-down (share MM3)")
td_mod  = w_de("top-down (share modelado)")
td_ren  = w_de("top-down (modelado + renorm)")

print()
print("VEREDICTO -- son DOS preguntas distintas y conviene no mezclarlas")
print()
print(f"(a) Sirve la DESCOMPOSICION?   top-down(MM3) {td_mm3:.4f} vs bottom-up {bu:.4f}")
d_a = 100*(bu - td_mm3)/bu
print(f"    -> {'SI' if d_a > 1 else ('NO' if d_a < -1 else 'EMPATE')}  ({d_a:+.1f}%)")
print()
print(f"(b) Sirve MODELAR el share?    modelado {td_mod:.4f} vs MM3 {td_mm3:.4f}")
d_b = 100*(td_mm3 - td_mod)/td_mm3
print(f"    -> {'SI' if d_b > 1 else ('NO' if d_b < -1 else 'EMPATE')}  ({d_b:+.1f}%)")
print(f"    WAPE del propio share en val: modelo {wape(va['y_share'], sh_va):.4f} "
      f"vs MM3 {wape(va['y_share'], sh_va_mm3):.4f}")
print()
print(f"(c) Sirve RENORMALIZAR?        renorm {td_ren:.4f} vs sin renorm {td_mod:.4f}")
d_c = 100*(td_mod - td_ren)/td_mod
print(f"    -> {'SI' if d_c > 1 else ('NO' if d_c < -1 else 'EMPATE')}  ({d_c:+.1f}%)")
print()
if d_a > 1 and d_b <= 1:
    print("LECTURA: la ganancia viene de modelar el TOTAL, no el share.")
    print("Es H1 otra vez -- el agregado es mas predecible -- y no la hipotesis de que")
    print("la redistribucion de share se explique por novedad y fase. El share no se")
    print("predice mejor que con su propia media movil, aun dandole edad, fase,")
    print("entradas de competidores y momentum del share.")
    print("La descomposicion igual conviene: anclar al total modelado gana "
          f"{d_a:.1f}% sobre predecir tn directo.")

In [ ]:
d = R.sort("wape_test")
fig, ax = plt.subplots(figsize=(8.5, 3.4))
ys = list(range(d.height))
cols = [SERIE[0] if "top-down" in n else (SERIE[1] if "bottom-up" in n else MUDO)
        for n in d["modelo"]]
ax.barh(ys, d["wape_test"], color=cols, height=.62)
for i, v in enumerate(d["wape_test"]):
    ax.annotate(f"{v:.4f}", (v, i), xytext=(6,0), textcoords="offset points",
                va="center", color=TINTA2, fontsize=9)
ax.set_yticks(ys); ax.set_yticklabels(d["modelo"])
ax.invert_yaxis(); ax.set_xlim(0, float(d["wape_test"].max())*1.2)
limpiar(ax, f"WAPE en test (mes {MESES_TEST[0]}), horizonte {H}", x="WAPE")
ax.grid(axis="y", visible=False); ax.grid(axis="x", visible=True)
plt.tight_layout(); plt.show()

## 5 — ¿Dónde gana cada uno?

Ésta es la pregunta que de verdad importa, y la que motivó el notebook: **la ventaja
del top-down debería concentrarse en los productos nuevos**, donde la curva de vida
es informativa y la serie propia es corta. En los maduros, con 24 meses de historia,
el bottom-up tiene todo lo que necesita.

Si el corte por edad y por fase muestra eso, la conclusión no es "gana uno u otro"
sino **un modelo híbrido**: top-down donde la historia es corta, bottom-up donde
sobra.

Mirá las dos tablas con atención, porque no dicen lo mismo. La **fase** suele ordenar
la ventaja mucho mejor que la **edad**, y tiene sentido: un producto de 30 meses que
entró en crecimiento se parece más a un lanzamiento que a uno de 30 meses en meseta.
La edad es una proxy cruda de lo que la fase mide directamente.

In [ ]:
pv_best = reconstruir(va, sh_va, tot_va, True)
pt_best = reconstruir(te, sh_te, tot_te, True)

ev = te.select("product_id","edad","fase","es_nuevo","y_tn").to_pandas()
ev["pred_bu"] = pred_bu_te
ev["pred_td"] = pt_best

def tabla(col, etiquetas=None):
    out = []
    for v, g in ev.groupby(col):
        if len(g) < 20:
            continue
        wbu = wape(g["y_tn"], g["pred_bu"]); wtd = wape(g["y_tn"], g["pred_td"])
        out.append({col: etiquetas.get(v, v) if etiquetas else v, "n": len(g),
                    "wape_bottomup": round(wbu,4), "wape_topdown": round(wtd,4),
                    "ventaja_topdown_%": round(100*(wbu-wtd)/wbu, 1)})
    return pl.DataFrame(out)

FASES = {0:"0_lanzamiento", 1:"1_crecimiento", 2:"2_meseta", 3:"3_caida"}
ev["tramo_edad"] = pd.cut(ev["edad"], [-1,5,11,23,999],
                          labels=["0-5 meses","6-11","12-23","24+"])

print("=== POR EDAD DEL PRODUCTO ===")
print(tabla("tramo_edad"))
print()
print("=== POR FASE ===")
print(tabla("fase", FASES))

In [ ]:
t = tabla("tramo_edad").sort("tramo_edad")
xs = np.arange(t.height); w = .38
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar(xs - w/2, t["wape_bottomup"], width=w, color=SERIE[1], label="bottom-up")
ax.bar(xs + w/2, t["wape_topdown"],  width=w, color=SERIE[0], label="top-down")
for i, (a, b) in enumerate(zip(t["wape_bottomup"], t["wape_topdown"])):
    ax.annotate(f"{a:.3f}", (i-w/2, a), xytext=(0,3), textcoords="offset points",
                ha="center", color=TINTA2, fontsize=8)
    ax.annotate(f"{b:.3f}", (i+w/2, b), xytext=(0,3), textcoords="offset points",
                ha="center", color=TINTA2, fontsize=8)
ax.set_xticks(xs); ax.set_xticklabels([str(v) for v in t["tramo_edad"]])
limpiar(ax, "WAPE por edad del producto", y="WAPE", x="edad al momento de predecir")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

gan = t.filter(pl.col("ventaja_topdown_%") > 0)
if gan.height:
    print("Tramos donde el top-down gana: " +
          ", ".join(f"{r['tramo_edad']} ({r['ventaja_topdown_%']:+.1f}%)" for r in gan.iter_rows(named=True)))
    print()
    print("Un modelo hibrido usaria top-down solo en esos tramos. La ganancia global")
    print("de hacerlo se estima abajo.")
else:
    print("El top-down no gana en ningun tramo de edad.")

In [ ]:
# ── Hibrido: top-down donde gano en VALIDACION, bottom-up en el resto ────
# Se prueban dos criterios de particion y gana el que mejor ande EN VALIDACION.
# Elegir el criterio mirando el test seria hacer trampa.
FASES_L = {0:"0_lanzamiento", 1:"1_crecimiento", 2:"2_meseta", 3:"3_caida"}

va_pd = va.select("edad","fase").to_pandas()
va_pd["y"] = va["y_tn"].to_numpy(); va_pd["bu"] = pred_bu_va; va_pd["td"] = pv_best
va_pd["tramo"] = pd.cut(va_pd["edad"], [-1,5,11,23,999], labels=["0-5","6-11","12-23","24+"])

ev["tramo"] = pd.cut(ev["edad"], [-1,5,11,23,999], labels=["0-5","6-11","12-23","24+"])

resumen_hib = []
for criterio in ("tramo", "fase"):
    usar_td = set()
    for v, g in va_pd.groupby(criterio, observed=True):
        if len(g) >= 20 and wape(g["y"], g["td"]) < wape(g["y"], g["bu"]):
            usar_td.add(v)
    hib_va = np.where(va_pd[criterio].isin(usar_td), va_pd["td"], va_pd["bu"])
    hib_te = np.where(ev[criterio].isin(usar_td), ev["pred_td"], ev["pred_bu"])
    etiquetas = [FASES_L.get(v, str(v)) for v in sorted(usar_td, key=str)]
    resumen_hib.append({
        "criterio": criterio,
        "grupos_con_topdown": ", ".join(etiquetas) or "ninguno",
        "wape_val":  round(wape(va_pd["y"], hib_va), 4),
        "wape_test": round(wape(ev["y_tn"], hib_te), 4),
    })

Hb = pl.DataFrame(resumen_hib)
w_bu = wape(ev["y_tn"], ev["pred_bu"])
w_td = wape(ev["y_tn"], ev["pred_td"])
Hb = Hb.with_columns((100*(w_bu - pl.col("wape_test"))/w_bu).round(2).alias("vs_bottomup_%"))
print(Hb)
print()
print(f"referencia   bottom-up puro  test {w_bu:.4f}")
print(f"             top-down puro   test {w_td:.4f}")
print()

# El criterio se elige por VALIDACION, no por test
elegido = Hb.sort("wape_val")["criterio"][0]
fila = Hb.filter(pl.col("criterio") == elegido).row(0, named=True)
print(f"criterio elegido en validacion: '{elegido}'  -> usa top-down en: {fila['grupos_con_topdown']}")
print(f"su resultado en TEST: {fila['wape_test']:.4f}  ({fila['vs_bottomup_%']:+.2f}% vs bottom-up)")
print()
print("ADVERTENCIA DE TAMANIO: el test es UN mes y unos cientos de filas por grupo.")
print("Diferencias de 1-3% con esa muestra son sugerentes, no concluyentes. Antes de")
print("decidir nada, repetir con walk-forward sobre varios meses de test.")

## 6 — Exportar

In [ ]:
# La prediccion top-down como FEATURE para el pipe, no como alternativa.
# Se genera para todas las filas con datos suficientes.
todo = D
sh_all = np.clip(m_sh.predict(a_pd(todo)), 0, None)
c_all = C
tot_all = dict(zip(zip(c_all["cat3"], c_all["m"]), total_pred(c_all)))
pred_all = reconstruir(todo, sh_all, tot_all, True)

export = todo.select("product_id","periodo").with_columns([
    pl.Series("pred_share_h2", sh_all),
    pl.Series("pred_topdown_h2", pred_all),
]).join(todo.select("product_id","periodo","edad","fase","es_nuevo",
                    "share_desvio","share_mom","entradas_6m","z_vs_ma12"),
        on=["product_id","periodo"], how="left")

out = DIR_OUT / "features_topdown.parquet"
export.write_parquet(out)
print(f"Guardado: {out}")
print(f"{export.height:,} filas x {export.width} columnas")
print(export.columns)

### Sobre estas features

`pred_topdown_h2` y `pred_share_h2` son **predicciones de un modelo entrenado acá**.
Al pegarlas al dataset del pipe hay un riesgo real: el modelo de share se entrenó con
meses que después van a estar en el train del pipe, así que para esas filas la
predicción es *in-sample* y se ve mejor de lo que sería en producción. El GBM del
pipe puede sobre-confiar en ella.

Dos formas de manejarlo:

- **La prolija**: regenerar estas features con validación cruzada temporal — predecir
  cada bloque de meses con un modelo entrenado sólo con meses anteriores.
- **La rápida**: pegar sólo las features *crudas* (`share_desvio`, `share_mom`,
  `entradas_6m`, `es_nuevo`, `fase`, `edad`) y dejar que el GBM del pipe arme la
  relación él mismo. Sin predicciones de por medio, no hay in-sample que filtre.

Empezaría por la rápida: son las que llevan la información de ciclo de vida y
competencia que motivó todo esto, sin el riesgo.